[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-og-helse/blob/main/uke07-velferdsteknologi/01_robotnavigasjon_i_rutenett_med_astar.ipynb)


# 🤖 Robotnavigasjon i 2D-grid med A*

## Læringsmål
- Forstå A* algoritmen og heuristikkens rolle
- Implementere rutefinning i et grid med hindringer
- Visualisere funnet sti i gridet


### 🔧 Miljøoppsett – fungerer både lokalt og i Google Colab


In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("🚀 Kjører i Google Colab")
    if not os.path.exists('AI-og-helse'):
        import subprocess
        subprocess.check_call(["git", "clone", "https://github.com/arvidl/AI-og-helse.git"])  # type: ignore
    if os.path.exists('AI-og-helse'):
        os.chdir('AI-og-helse')
        print(f"📁 Byttet til mappe: {os.getcwd()}")
else:
    print("💻 Kjører i lokal miljø")

import numpy as np
import matplotlib.pyplot as plt
print("✅ Miljø klart")


In [ ]:
import heapq

# A* i grid med 4-naboer

def astar(grid: np.ndarray, start: tuple[int,int], goal: tuple[int,int]):
    def in_bounds(x, y):
        return 0 <= x < grid.shape[0] and 0 <= y < grid.shape[1]

    def neighbors(x, y):
        for dx, dy in [(1,0),(-1,0),(0,1),(0,-1)]:
            nx, ny = x+dx, y+dy
            if in_bounds(nx, ny) and grid[nx, ny] == 0:
                yield (nx, ny)

    def manhattan(a, b):
        return abs(a[0]-b[0]) + abs(a[1]-b[1])

    open_heap = [(0, start)]
    g = {start: 0}
    came_from = {start: None}

    while open_heap:
        _, cur = heapq.heappop(open_heap)
        if cur == goal:
            # rekonstruksjon
            path = []
            while cur is not None:
                path.append(cur)
                cur = came_from[cur]
            return path[::-1]
        for nb in neighbors(*cur):
            tentative = g[tuple(cur)] + 1
            if tentative < g.get(tuple(nb), float('inf')):
                g[tuple(nb)] = tentative
                came_from[tuple(nb)] = tuple(cur)
                f = tentative + manhattan(nb, goal)
                heapq.heappush(open_heap, (f, tuple(nb)))
    return None


In [ ]:
# Demo-grid
np.random.seed(0)
H, W = 15, 25
grid = np.zeros((H, W), dtype=int)
# legg inn tilfeldige hindringer
for _ in range(80):
    grid[np.random.randint(0, H), np.random.randint(0, W)] = 1
# sørg for at start/goal er fri
start = (0, 0)
goal = (H-1, W-1)
grid[start] = 0
grid[goal] = 0

path = astar(grid, start, goal)
print("Fant sti:", path is not None)

# Visualisering
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(grid, cmap='gray_r')
if path:
    xs = [p[1] for p in path]
    ys = [p[0] for p in path]
    ax.plot(xs, ys, color='red', linewidth=2)
ax.scatter([start[1], goal[1]], [start[0], goal[0]], c=['green','blue'], s=80)
ax.set_title('A* i 2D-grid')
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


### Refleksjon
- Hvordan påvirker heuristikk (Manhattan vs. Euclidean) kjøretid og optimalitet?
- Hva skjer når gridet er tett av hindringer?
- Hvordan ville du utvidet til 8-naboer eller kostnader per celle?
